# Part 3：Nsight GPU Metrics 跨裝置模型

## 研究問題

不使用來源 baseline runtime，只使用 marker-aligned GPU behavior，能否預測 target runtime？

## 目前結論

compact_gpu_ridge 使用 15 個 features，combined MAPE 3.70%、R² 0.9915。它證明 GPU behavior 有訊號，但 segmentation 仍依賴 marker。

> 本 Notebook 新增於既有研究產物之外，不會取代原始 Python scripts 或 CSV／Markdown 報告。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts at the repo or notebook folder."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "analysis").is_dir() and (candidate / "analysis_outputs").is_dir():
            return candidate
    raise FileNotFoundError("找不到專案根目錄；請從本 repository 內啟動 Jupyter。")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "analysis_outputs"
RUN_EXPENSIVE = False  # 明確改成 True 才會重跑分析腳本。


def project_path(relative: str) -> Path:
    return PROJECT_ROOT / relative


def load_csv(relative: str, **kwargs) -> pd.DataFrame:
    path = project_path(relative)
    if not path.exists():
        raise FileNotFoundError(f"缺少資料：{path}")
    return pd.read_csv(path, **kwargs)


def show_markdown(relative: str) -> None:
    path = project_path(relative)
    display(Markdown(path.read_text(encoding="utf-8")))


def inspect_inputs(paths) -> pd.DataFrame:
    rows = []
    for relative in paths:
        path = project_path(relative)
        rows.append({
            "path": relative,
            "exists": path.exists(),
            "type": "directory" if path.is_dir() else "file",
            "size_mib": round(path.stat().st_size / 1024**2, 3) if path.is_file() else None,
        })
    return pd.DataFrame(rows)


def run_commands(commands) -> None:
    """Show commands by default; run only after explicit opt-in."""
    for args in commands:
        command = [sys.executable, *args]
        print(" ".join(map(str, command)))
        if RUN_EXPENSIVE:
            subprocess.run(command, cwd=PROJECT_ROOT, check=True)


print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RUN_EXPENSIVE = {RUN_EXPENSIVE}")

## 實驗步驟

1. 從 Nsight SQLite 匯出 GPU metric samples。
2. 以 marker 對齊 steady window。
3. 建立 static、duration 與 compact GPU features。
4. 在 nested direction-specific LOOW 中選 Ridge alpha。
5. 只以 target baseline 作 label，產生 OOF predictions。

## 輸入與 provenance

下列 cell 只檢查輸入是否存在，不修改資料。

In [ ]:
INPUTS = [
    "analysis_outputs/nsys_gpu_metrics_features.csv",
    "analysis_outputs/workload_features.csv",
    "hardware/hardware_descriptors.csv",
]
display(inspect_inputs(INPUTS))

## 重現入口

下列命令依現有 pipeline 排列。預設只列印；將 RUN_EXPENSIVE 明確改成 True 才會執行。

In [ ]:
COMMANDS = [
    ["analysis/extract_nsys_gpu_metrics.py"],
    ["analysis/run_profile_models.py"],
]
run_commands(COMMANDS)

## 結果表

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part3_profile_models/metrics.csv"
PREDICTIONS_PATH = "analysis_outputs/model_experiments/part3_profile_models/predictions.csv"

metrics = load_csv(METRICS_PATH)
sort_column = "mape_percent" if "mape_percent" in metrics.columns else "mean_period_ape_percent"
display(metrics.sort_values(sort_column).reset_index(drop=True))

if "direction" in metrics.columns and "model" in metrics.columns:
    plot_data = metrics.sort_values(sort_column).groupby("direction", as_index=False).first()
    ax = plot_data.plot.bar(x="direction", y=sort_column, legend=False, figsize=(8, 4))
    ax.set_ylabel(sort_column)
    ax.set_title("各方向最佳列（依表內指標排序）")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## OOF 預測與 worst cases

顯示現有 prediction table 中 APE 最大的列，並提供 actual-vs-predicted 圖。

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part3_profile_models/metrics.csv"
PREDICTIONS_PATH = "analysis_outputs/model_experiments/part3_profile_models/predictions.csv"

predictions = load_csv(PREDICTIONS_PATH)
display(predictions.sort_values("ape_percent", ascending=False).head(15).reset_index(drop=True))

required = {"actual_ms", "predicted_ms", "ape_percent"}
if required.issubset(predictions.columns):
    view = predictions.copy()
    if "deployment_valid" in view.columns:
        view = view[view["deployment_valid"].astype(str).str.lower().eq("true")]
    ax = view.plot.scatter(x="actual_ms", y="predicted_ms", c="ape_percent", cmap="viridis", figsize=(6, 5))
    low = min(view["actual_ms"].min(), view["predicted_ms"].min())
    high = max(view["actual_ms"].max(), view["predicted_ms"].max())
    ax.plot([low, high], [low, high], "--", color="black", linewidth=1)
    ax.set_title("OOF prediction：actual vs predicted")
    plt.tight_layout()
    plt.show()

## 既有最終報告快照

> 來源：`analysis_outputs/model_experiments/part3_profile_models/report.md`。這是建立 Notebook 時嵌入的結果快照；重新執行分析後，請以磁碟上的最新報告為準。

# Part 3: deployment-aligned Nsight GPU profiling models

The target-device unprofiled baseline runtime is used only as the label. Model inputs contain only workload fields and source-device Nsight GPU profiling fields available before prediction. `source_ground_truth_iter_ms` is excluded from every Part 3 model.

Common complete GPU summary inputs: 66 columns (mean, CV, and P90 summaries).

Outer evaluation is leave-one-workload-out. Variance filtering, scaling, PCA, and hyperparameter selection are fitted only inside each outer training fold.

| Direction | Model | Inputs | MAE ms | RMSE ms | MAPE | Median APE | Max APE | R2 |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| RTX4090_to_RTX5090 | compact_gpu_ridge | 15 | 1.284 | 1.549 | 3.14% | 2.17% | 10.74% | 0.9935 |
| RTX4090_to_RTX5090 | static_gpu_summary_pca_ridge | 70 | 4.212 | 14.227 | 5.99% | 3.28% | 66.41% | 0.4530 |
| RTX4090_to_RTX5090 | profile_duration_proxy_ridge | 4 | 2.998 | 4.332 | 6.31% | 4.93% | 16.94% | 0.9493 |
| RTX4090_to_RTX5090 | static_gpu_summary_ridge | 70 | 4.626 | 13.339 | 6.95% | 4.89% | 61.42% | 0.5191 |
| RTX4090_to_RTX5090 | gpu_summary_pca_ridge | 67 | 4.986 | 16.515 | 7.07% | 3.34% | 76.75% | 0.2628 |
| RTX4090_to_RTX5090 | static_only_ridge | 3 | 6.966 | 10.439 | 14.45% | 13.51% | 38.35% | 0.7055 |
| RTX5090_to_RTX4090 | compact_gpu_ridge | 15 | 2.028 | 2.429 | 4.26% | 3.25% | 10.22% | 0.9900 |
| RTX5090_to_RTX4090 | static_gpu_summary_ridge | 70 | 2.642 | 3.668 | 4.66% | 4.15% | 9.56% | 0.9771 |
| RTX5090_to_RTX4090 | gpu_summary_pca_ridge | 67 | 2.580 | 3.546 | 4.67% | 3.70% | 11.03% | 0.9786 |
| RTX5090_to_RTX4090 | static_gpu_summary_pca_ridge | 70 | 2.666 | 3.881 | 4.73% | 3.84% | 11.39% | 0.9744 |
| RTX5090_to_RTX4090 | profile_duration_proxy_ridge | 4 | 3.527 | 4.708 | 7.30% | 6.24% | 16.47% | 0.9623 |
| RTX5090_to_RTX4090 | static_only_ridge | 3 | 10.976 | 16.141 | 19.72% | 18.53% | 44.20% | 0.5574 |
| both_directions_combined | compact_gpu_ridge | 15 | 1.656 | 2.037 | 3.70% | 2.58% | 10.74% | 0.9915 |
| both_directions_combined | static_gpu_summary_pca_ridge | 70 | 3.439 | 10.428 | 5.36% | 3.62% | 66.41% | 0.7771 |
| both_directions_combined | static_gpu_summary_ridge | 70 | 3.634 | 9.782 | 5.80% | 4.39% | 61.42% | 0.8039 |
| both_directions_combined | gpu_summary_pca_ridge | 67 | 3.783 | 11.944 | 5.87% | 3.34% | 76.75% | 0.7076 |
| both_directions_combined | profile_duration_proxy_ridge | 4 | 3.263 | 4.524 | 6.80% | 5.56% | 16.94% | 0.9581 |
| both_directions_combined | static_only_ridge | 3 | 8.971 | 13.592 | 17.08% | 16.15% | 44.20% | 0.6213 |

## Comparison with earlier idealized baselines

Part 1 and Part 2 use the source device's unprofiled ground-truth runtime, which is a stronger input than the short-profiling deployment scenario. They are reported as reference upper baselines, not deployment-equivalent competitors.

| Direction | Part 1 best | Part 1 MAPE | Part 2 best | Part 2 MAPE | Part 3 best | Part 3 MAPE |
|---|---|---:|---|---:|---|---:|
| RTX4090_to_RTX5090 | log_source_ols | 4.10% | huber_interactions_log_target | 3.11% | compact_gpu_ridge | 3.14% |
| RTX5090_to_RTX4090 | global_ratio_median | 4.23% | ridge_main_log_target | 4.27% | compact_gpu_ridge | 4.26% |

## Hyperparameter selection frequency

- RTX4090_to_RTX5090 / compact_gpu_ridge: `{"model__alpha": 0.1}`: 18; `{"model__alpha": 0.01}`: 6
- RTX4090_to_RTX5090 / gpu_summary_pca_ridge: `{"model__alpha": 1.0, "pca__n_components": 12}`: 16; `{"model__alpha": 1.0, "pca__n_components": 16}`: 3; `{"model__alpha": 0.1, "pca__n_components": 8}`: 3; `{"model__alpha": 0.1, "pca__n_components": 12}`: 1; `{"model__alpha": 10.0, "pca__n_components": 12}`: 1
- RTX4090_to_RTX5090 / profile_duration_proxy_ridge: `{"model__alpha": 1.0}`: 17; `{"model__alpha": 0.1}`: 7
- RTX4090_to_RTX5090 / static_gpu_summary_pca_ridge: `{"model__alpha": 1.0, "pca__n_components": 12}`: 11; `{"model__alpha": 0.1, "pca__n_components": 8}`: 7; `{"model__alpha": 0.1, "pca__n_components": 12}`: 4; `{"model__alpha": 1.0, "pca__n_components": 16}`: 1; `{"model__alpha": 1.0, "pca__n_components": 8}`: 1
- RTX4090_to_RTX5090 / static_gpu_summary_ridge: `{"model__alpha": 1.0}`: 10; `{"model__alpha": 0.1}`: 8; `{"model__alpha": 10.0}`: 3; `{"model__alpha": 0.01}`: 3
- RTX4090_to_RTX5090 / static_only_ridge: `{"model__alpha": 0.01}`: 15; `{"model__alpha": 1.0}`: 7; `{"model__alpha": 0.1}`: 2
- RTX5090_to_RTX4090 / compact_gpu_ridge: `{"model__alpha": 0.1}`: 21; `{"model__alpha": 0.01}`: 3
- RTX5090_to_RTX4090 / gpu_summary_pca_ridge: `{"model__alpha": 1.0, "pca__n_components": 16}`: 17; `{"model__alpha": 0.1, "pca__n_components": 16}`: 6; `{"model__alpha": 1.0, "pca__n_components": 8}`: 1
- RTX5090_to_RTX4090 / profile_duration_proxy_ridge: `{"model__alpha": 1.0}`: 24
- RTX5090_to_RTX4090 / static_gpu_summary_pca_ridge: `{"model__alpha": 1.0, "pca__n_components": 16}`: 13; `{"model__alpha": 0.1, "pca__n_components": 16}`: 6; `{"model__alpha": 0.1, "pca__n_components": 8}`: 2; `{"model__alpha": 1.0, "pca__n_components": 8}`: 2; `{"model__alpha": 0.1, "pca__n_components": 12}`: 1
- RTX5090_to_RTX4090 / static_gpu_summary_ridge: `{"model__alpha": 1.0}`: 16; `{"model__alpha": 0.1}`: 7; `{"model__alpha": 0.01}`: 1
- RTX5090_to_RTX4090 / static_only_ridge: `{"model__alpha": 1.0}`: 13; `{"model__alpha": 10.0}`: 10; `{"model__alpha": 0.1}`: 1

## Worst workloads for the best Part 3 model

### RTX4090_to_RTX5090: `compact_gpu_ridge`

| Workload | Actual ms | Predicted ms | APE |
|---|---:|---:|---:|
| C01 | 29.060 | 25.940 | 10.74% |
| C16 | 41.930 | 39.087 | 6.78% |
| C13 | 30.371 | 32.367 | 6.57% |
| C02 | 28.075 | 29.809 | 6.18% |
| C09 | 30.749 | 32.236 | 4.84% |

### RTX5090_to_RTX4090: `compact_gpu_ridge`

| Workload | Actual ms | Predicted ms | APE |
|---|---:|---:|---:|
| C10 | 49.200 | 44.170 | 10.22% |
| C17 | 38.002 | 34.560 | 9.06% |
| C09 | 31.867 | 34.640 | 8.70% |
| C06 | 34.708 | 37.431 | 7.85% |
| C21 | 35.460 | 37.809 | 6.62% |


## 解讀檢查清單

- 區分 OOF evaluation 與重新用全部資料訓練的 production bundle。
- 不把 target runtime、oracle marker 或 future information 當作 deployment feature。
- 報告方向別結果、combined summary 與 worst cases。
- 結論只適用於目前資料、GPU 節點與 workload family。